# Policy Iteration
Реализовать Policy Iteration, выполнить задания:

1. В алгоритме Policy Iteration важным гиперпараметром является gamma. Требуется ответить на вопрос, какой gamma лучше выбирать. Качество обученной политики можно оценивать например запуская среду 1000 раз и взяв после этого средний total_reward. 
2. На шаге Policy Evaluation мы каждый раз начинаем с нулевых values. А что будет если вместо этого начинать с values обученных на предыдущем шаге? Будет ли алгоритм работать? Если да, то будет ли он работать лучше? 

# Libraries

In [2]:
import numpy as np
from Frozen_Lake import FrozenLakeEnv

## Policy

In [11]:
class Policy:
    def __init__(self, env: FrozenLakeEnv):
        self.env = env
        self.policy = {}
        self._init()
    
    def _init(self) -> None:
        for state in self.env.get_all_states():
            actions = self.env.get_possible_actions(state)
            if not actions:
                actions = {None: 1}
            uniform = 1 / len(actions)
            self.policy[state] = {
                action: uniform 
                for action in actions
            }
            
    def reset(self) -> None: 
        self.policy.clear()
        self._init()
            
    def action(self, state: tuple) -> str:
        pass

## V & Q functions

In [67]:
def init_v(env: FrozenLakeEnv) -> dict:
    return {  
        state: 0 
        for state in env.get_all_states()
    }


def compute_v(
        env: FrozenLakeEnv, 
        q: dict, 
        policy: Policy
) -> dict:
    new_v = init_v(env)
    for state in env.get_all_states():
        new_v[state] = 0
        for action in env.get_possible_actions(state):
            new_v[state] += policy.policy[state][action] * q[state][action]
    return new_v


def compute_q(
        env: FrozenLakeEnv, 
        v: dict, 
        gamma: float
) -> dict:
    q = {}
    for state in env.get_all_states():
        q[state] = {}
        for action in env.get_possible_actions(state):
            q[state][action] = 0
            for next_state in env.get_next_states(state, action):
                transition_prob = env.get_transition_prob(state, action, next_state)
                reward = env.get_reward(state, action, next_state)
                q[state][action] += (transition_prob * reward) + (transition_prob * gamma * v[next_state])
    return q


# Eval & improvefor state in env.get_all_states():
            v

In [102]:
def evaluation(
        env: FrozenLakeEnv,
        policy: Policy,
        gamma: float, 
        eval_iter: int,
        v=None
) -> dict:
    v = v if v else init_v(env)
    for _ in range(eval_iter):
        q = compute_q(env, v, gamma)
        v = compute_v(env, q, policy)
    return v, compute_q(env, v, gamma)


def improvement(q_values):
    policy = {}
    for state in env.get_all_states():
        policy[state] = {}
        argmax_action = None
        max_q_value = float('-inf')
        for action in env.get_possible_actions(state): 
            policy[state][action] = 0
            if q_values[state][action] > max_q_value:
                argmax_action = action
                max_q_value = q_values[state][action]
        policy[state][argmax_action] = 1
        
    _policy = Policy(env)
    _policy.policy = policy
    return _policy

# Train

In [103]:
def train(env, policy, epoch, gamma, eval_iter):
    v = init_v(env)
    for _ in range(epoch):
        v, q = evaluation(env, policy, gamma, eval_iter, v)
        policy = improvement(q)
    return policy
    
    
def _eval(env, policy) -> float:
    total_rewards = []
    
    for _ in range(1_000):
        total_reward = 0
        state = env.reset()
        
        for _ in range(100):
            actions = env.get_possible_actions(state)
            p = list(policy.policy[state].values())
            action = np.random.choice(actions, p=p)
    
            state, reward, done, _ = env.step(action)
            total_reward += reward
            
            if done:
                break
        
        total_rewards.append(total_reward)
    
    return np.mean(total_rewards)

# Check results

In [80]:
env = FrozenLakeEnv()
policy = Policy(env)


for gamma in np.linspace(0.8, 1.0, 10):
    policy = train(env, policy, 100, gamma, 100)
    MTR = _eval(env, policy)
    print(f"gamma: {round(gamma, 2)} MTR: {round(MTR, 2)}")

gamma: 0.8 MTR: 0.72
gamma: 0.82 MTR: 0.73
gamma: 0.84 MTR: 0.76
gamma: 0.87 MTR: 0.77
gamma: 0.89 MTR: 0.68
gamma: 0.91 MTR: 0.73
gamma: 0.93 MTR: 0.7
gamma: 0.96 MTR: 0.66
gamma: 0.98 MTR: 0.83
gamma: 1.0 MTR: 0.84


In [108]:
gammaenv = FrozenLakeEnv()
policy = Policy(env)


policy = train(env, policy, 10, 0.99, 100)
MTR = _eval(env, policy)
MTR

0.851